# DeepSeek R1 70B Q4 Inference on Google Colab A100

**目标**: 在 Colab A100 (80GB) 上运行 DeepSeek R1 Distill 70B Q4 推理，生成完整思维链用于 finetune Qwen3 7B

**特性**:
- vLLM 后端，支持 continuous batching
- 断点续传 (checkpoint/resume)
- 逐条保存结果
- 支持年份筛选 + 黄金切片法

**预估速度**: A100 80GB + 70B Q4 + vLLM batch=4 → ~15-20s/prompt → AAPL 28K条约 3-4 天

## 0. 检查 GPU 和运行时

In [ ]:
# 确认 Colab 分配了 A100 GPU
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    assert gpu_mem > 70, f"Need A100 80GB, got {gpu_mem:.0f}GB. Change runtime to A100!"

## 1. 安装依赖

In [ ]:
# 安装 vLLM (支持 A100 的 CUDA 版本)
!pip install vllm -q
!pip install huggingface_hub -q
print("\n✅ Dependencies installed")

## 2. 下载模型

从 HuggingFace 下载 DeepSeek R1 Distill 70B Q4 GGUF 模型。

⚠️ 首次下载约 40GB，需要 15-30 分钟。下载完后会缓存在 Colab 的磁盘上。

In [ ]:
# ===== 模型选择 =====
# 方案A: 使用 HuggingFace 原始权重 (FP16, ~140GB, 需要量化)
# 方案B: 使用 vLLM 直接加载 AWQ/GPTQ 量化模型 (推荐)

# 推荐: 使用 AWQ 量化的 70B 模型 (~38GB, 适合 A100 80GB)
MODEL_ID = "casperhansen/deepseek-r1-distill-qwen-70b-awq"

# 备选: 如果上面的不可用，使用 GPTQ 版本
# MODEL_ID = "TheBloke/deepseek-r1-distill-qwen-70b-GPTQ"

# 备选: 原始 FP16 (需要 A100 80GB 的全部显存，可能不够 KV cache)
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-70B"

print(f"Model: {MODEL_ID}")
print(f"This will be downloaded on first use by vLLM (~38GB for AWQ)")

## 3. 上传 Prompt 数据

上传你的 JSONL prompt 文件到 Colab。

**两种方式**:
- A) 从 Google Drive 挂载
- B) 直接上传文件

In [ ]:
# ===== 方式A: Google Drive (推荐，数据持久化) =====
from google.colab import drive
drive.mount('/content/drive')

# 设置数据目录 - 修改为你的 Drive 路径
DRIVE_DATA_DIR = "/content/drive/MyDrive/Taihang-llm/data"

import os
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print(f"Data directory: {DRIVE_DATA_DIR}")
print(f"Files found: {[f for f in os.listdir(DRIVE_DATA_DIR) if f.endswith('.jsonl')]}")

In [ ]:
# ===== 方式B: 直接上传文件 (适合少量文件) =====
# 取消注释下面的代码来使用文件上传

# from google.colab import files
# uploaded = files.upload()  # 会弹出文件选择对话框
# !mkdir -p /content/data
# !mv *.jsonl /content/data/
# DRIVE_DATA_DIR = "/content/data"

## 4. 配置参数

In [ ]:
# ===== 推理配置 =====

# 要处理的 symbol (一次处理一个，避免超时)
SYMBOL = "AAPL"

# 年份筛选 (可选)
# 设为 None 处理全部年份
# 设为列表只处理指定年份: [2020, 2021, 2022, 2023, 2024]
YEARS_SELECT = None

# 黄金切片法 (可选)
# 设为 None 不使用
# 设为数字自动选择 N 个代表性年份: 5
GOLDEN_SLICE = None  # e.g., 5 = 从15年中选5个代表性年份

# 推理参数
MAX_TOKENS = 4096      # 保留完整思维链
TEMPERATURE = 0.7
TOP_P = 0.9

# vLLM 参数
TENSOR_PARALLEL = 1    # A100 单卡
GPU_MEMORY_UTIL = 0.92 # 使用 92% 显存
MAX_MODEL_LEN = 8192   # 最大上下文长度

# 路径
DATA_DIR = DRIVE_DATA_DIR
OUTPUT_DIR = os.path.join(DATA_DIR, "reasoning_output")
CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Symbol: {SYMBOL}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## 5. 启动 vLLM 引擎

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading model: {MODEL_ID}")
print(f"This may take 5-10 minutes on first run (downloading + loading weights)...")

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    quantization="awq",  # 如果用 GPTQ 模型改为 "gptq"
    dtype="float16",
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
)

print(f"\n✅ Model loaded successfully!")
print(f"GPU memory after loading:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6. 加载数据 + 年份筛选

In [ ]:
import json
import re
import glob

def golden_section_years(all_years, n_select):
    """黄金切片法选择代表性年份"""
    if n_select >= len(all_years):
        return all_years
    PHI = 1.6180339887
    total = len(all_years)
    indices = {0, total - 1}
    for i in range(1, n_select * 3):
        idx = int((i * PHI * total / n_select) % total)
        indices.add(idx)
        if len(indices) >= n_select:
            break
    while len(indices) < n_select:
        step = total / (n_select - len(indices) + 1)
        for i in range(total):
            indices.add(int(i * step) % total)
            if len(indices) >= n_select:
                break
    return sorted([all_years[i] for i in sorted(indices)[:n_select]])


# Load prompt file
prompt_file = glob.glob(os.path.join(DATA_DIR, f"deepseek_r1_input_prompts_{SYMBOL}_*.jsonl"))
if not prompt_file:
    raise FileNotFoundError(f"No prompt file found for {SYMBOL} in {DATA_DIR}")

prompt_file = prompt_file[0]
print(f"Loading: {os.path.basename(prompt_file)}")

prompts = []
with open(prompt_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            prompts.append(json.loads(line))

print(f"Loaded {len(prompts):,} prompts")

# Extract available years
year_counts = {}
for r in prompts:
    m = re.search(r'_(\d{4})-', r.get('custom_id', ''))
    if m:
        y = int(m.group(1))
        year_counts[y] = year_counts.get(y, 0) + 1

print(f"\nYear distribution:")
for y in sorted(year_counts):
    print(f"  {y}: {year_counts[y]:,} prompts")

# Apply year filter
available_years = sorted(year_counts.keys())

if GOLDEN_SLICE:
    selected_years = golden_section_years(available_years, GOLDEN_SLICE)
    print(f"\n🔶 Golden slice selected {len(selected_years)} years: {selected_years}")
elif YEARS_SELECT:
    selected_years = [y for y in YEARS_SELECT if y in available_years]
    print(f"\n📅 Manual year selection: {selected_years}")
else:
    selected_years = None
    print(f"\n📅 Processing ALL years")

if selected_years:
    year_set = set(str(y) for y in selected_years)
    prompts = [r for r in prompts
               if re.search(r'_(\d{4})-', r.get('custom_id', ''))
               and re.search(r'_(\d{4})-', r['custom_id']).group(1) in year_set]
    print(f"After filter: {len(prompts):,} prompts")

print(f"\n✅ Ready to process {len(prompts):,} prompts for {SYMBOL}")

## 7. 运行推理 (带断点续传)

In [ ]:
import time
from datetime import datetime
from tqdm.notebook import tqdm

# ===== Checkpoint logic (双重恢复: checkpoint文件 + 输出文件扫描) =====
ckpt_file = os.path.join(CHECKPOINT_DIR, f"ckpt_{SYMBOL}.json")
output_file = os.path.join(OUTPUT_DIR, f"deepseek_r1_reasoning_{SYMBOL}.jsonl")

completed_ids = set()

# Source 1: Load checkpoint JSON
if os.path.exists(ckpt_file):
    try:
        with open(ckpt_file, 'r') as f:
            ckpt_ids = set(json.load(f).get('completed_ids', []))
        completed_ids.update(ckpt_ids)
        print(f"📋 Checkpoint file: {len(ckpt_ids)} completed IDs")
    except Exception as e:
        print(f"⚠️ Failed to load checkpoint: {e}")

# Source 2: Scan existing output file (the ground truth — survives checkpoint loss)
if os.path.exists(output_file):
    output_ids = set()
    with open(output_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    record = json.loads(line)
                    cid = record.get('custom_id', '')
                    if cid:
                        output_ids.add(cid)
                except json.JSONDecodeError:
                    continue
    completed_ids.update(output_ids)
    print(f"📄 Output file scan: {len(output_ids)} completed results found")

if completed_ids:
    print(f"🔄 Total resuming: {len(completed_ids)} prompts already done (will skip)")
    # Sync checkpoint with output scan results
    with open(ckpt_file, 'w') as f:
        json.dump({'symbol': SYMBOL, 'completed_ids': list(completed_ids),
                   'count': len(completed_ids), 'updated_at': datetime.now().isoformat()}, f)

# Parse prompts and filter completed
tasks = []
for idx, record in enumerate(prompts, 1):
    custom_id = record.get('custom_id', f"{SYMBOL}_{idx}")
    if custom_id in completed_ids:
        continue
    body = record.get('body', {})
    msgs = body.get('messages', [])
    if msgs:
        prompt_text = msgs[-1].get('content', '')
    else:
        prompt_text = record.get('input_prompt', str(record))
    tasks.append((idx, custom_id, prompt_text))

total_all = len(prompts)
total_remaining = len(tasks)
print(f"\n📊 {SYMBOL}: {total_remaining} remaining / {total_all} total ({len(completed_ids)} done)")

if total_remaining == 0:
    print("✅ All prompts already completed!")
else:
    # ===== vLLM Batch Inference =====
    # vLLM 的 continuous batching 会自动优化吞吐量
    BATCH_SIZE = 32  # vLLM 内部会自动管理 GPU 并发

    success_count = 0
    fail_count = 0
    start_time = time.time()

    # Process in batches for better throughput
    pbar = tqdm(range(0, len(tasks), BATCH_SIZE), desc=f"Inferencing {SYMBOL}",
                total=(len(tasks) + BATCH_SIZE - 1) // BATCH_SIZE, unit="batch")

    for batch_start in pbar:
        batch = tasks[batch_start:batch_start + BATCH_SIZE]

        # Build conversation prompts for vLLM
        conversations = []
        for _, _, prompt_text in batch:
            conversations.append([
                {"role": "system", "content": "You are a specialized trading AI for financial analysis."},
                {"role": "user", "content": prompt_text}
            ])

        try:
            # vLLM batch generation - much faster than sequential
            outputs = llm.chat(conversations, sampling_params=sampling_params)

            # Save results
            for (idx, custom_id, prompt_text), output in zip(batch, outputs):
                reasoning = output.outputs[0].text if output.outputs else None
                if reasoning:
                    result = {
                        "symbol": SYMBOL,
                        "custom_id": custom_id,
                        "prompt": prompt_text[:500] + "..." if len(prompt_text) > 500 else prompt_text,
                        "reasoning": reasoning,
                    }
                    # Append to output file
                    with open(output_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(result, ensure_ascii=False) + '\n')
                    completed_ids.add(custom_id)
                    success_count += 1
                else:
                    fail_count += 1

            # Save checkpoint after each batch
            with open(ckpt_file, 'w') as f:
                json.dump({
                    'symbol': SYMBOL,
                    'completed_ids': list(completed_ids),
                    'count': len(completed_ids),
                    'updated_at': datetime.now().isoformat()
                }, f)

        except Exception as e:
            print(f"\n❌ Batch error: {e}")
            fail_count += len(batch)
            continue

        elapsed = time.time() - start_time
        speed = success_count / elapsed if elapsed > 0 else 0
        eta_s = (total_remaining - success_count - fail_count) / speed if speed > 0 else 0
        pbar.set_postfix(ok=success_count, fail=fail_count,
                         speed=f"{speed:.1f}/s", eta=f"{eta_s/3600:.1f}h")

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ {SYMBOL} Complete!")
    print(f"   Success: {success_count} | Failed: {fail_count}")
    print(f"   Time: {elapsed/3600:.1f} hours ({elapsed:.0f}s)")
    print(f"   Speed: {success_count/elapsed:.2f} prompts/sec")
    print(f"   Output: {output_file}")
    print(f"{'='*60}")

## 8. 查看结果

In [ ]:
# 查看输出文件
if os.path.exists(output_file):
    line_count = sum(1 for _ in open(output_file, encoding='utf-8'))
    file_size = os.path.getsize(output_file) / (1024 * 1024)
    print(f"Output file: {output_file}")
    print(f"Records: {line_count:,}")
    print(f"File size: {file_size:.1f} MB")
    
    # Show first result
    print(f"\n--- First result preview ---")
    with open(output_file, 'r', encoding='utf-8') as f:
        first = json.loads(f.readline())
    print(f"Symbol: {first['symbol']}")
    print(f"Custom ID: {first['custom_id']}")
    print(f"Reasoning length: {len(first['reasoning'])} chars")
    print(f"Reasoning preview: {first['reasoning'][:500]}...")
else:
    print("No output file yet. Run the inference cell first.")

## 9. 下载结果

In [ ]:
# 如果使用 Google Drive，结果已经自动保存在 Drive 中
# 如果需要直接下载到本地:

# from google.colab import files
# files.download(output_file)

## 10. 处理下一个 Symbol

修改上面的 `SYMBOL` 变量，然后从 **Cell 6** 重新开始运行。

模型已经加载在显存中，不需要重新加载。

In [ ]:
# 快捷方式: 批量处理多个 symbols
# 取消注释并运行此 cell 来自动处理多个 symbols

# SYMBOLS_TO_PROCESS = ["AAPL", "MSFT", "NVDA", "TSLA", "AMD"]
# for sym in SYMBOLS_TO_PROCESS:
#     SYMBOL = sym
#     print(f"\n{'='*60}")
#     print(f"Processing {SYMBOL}...")
#     # Copy-paste the loading + inference cells here or use %run